# Mock 09 — corrected notebook

Same task as the mock: 24h-ahead consumption forecast, Ridge, tuning, walk-forward.
Each **Fix:** note says what changed and why. Run this only after attempting the mock blind.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, KFold, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

pd.set_option("display.width", 120)

def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

from sklearn.pipeline import Pipeline, make_pipeline

In [2]:
df = pd.read_csv("../../data/hourly_power_clean.csv", parse_dates=["time"]).set_index("time")
y = df["consumption_mwh"]

**Fix 1 — features available at issuance time.** The forecast for hour *t* is issued at *t−24h*,
so `lag1` and `lag6` (consumption at *t−1h*, *t−6h*) do not exist yet. Only lags ≥ 24h and
rolling windows that end at *t−24h* are legitimate.

**Fix 2 — one-hot with `drop_first=True`.** 24 dummies plus an intercept are perfectly collinear;
Ridge still fits but the intercept is no longer "baseline load" and individual dummy coefficients
are not identified. Dropping one level restores interpretability.

In [3]:
feat = pd.DataFrame(index=df.index)
feat["lag24"] = y.shift(24)
feat["lag48"] = y.shift(48)
feat["lag168"] = y.shift(168)
feat["roll24"] = y.shift(24).rolling(24).mean()
feat["roll168"] = y.shift(24).rolling(168).mean()
feat["temp"] = df["temp_c"]
feat["dow"] = df.index.dayofweek
hours = pd.get_dummies(df.index.hour, prefix="h", drop_first=True).astype(float)
hours.index = df.index
feat = pd.concat([feat, hours], axis=1)

data = pd.concat([feat, y.rename("target")], axis=1).dropna()
X_raw, y_all = data.drop(columns="target"), data["target"]

# the leaked versions, kept only to quantify what they did in the mock
leaked = pd.concat([X_raw, y.shift(1).rename("lag1"), y.shift(6).rename("lag6")], axis=1).loc[X_raw.index]
X_raw.shape

(17329, 30)

**Fix 3 — scaler inside a Pipeline.** `StandardScaler().fit_transform(X)` on the full frame uses
test-period means/stds. Inside a `Pipeline` the scaler is refit on each training fold only.

In [4]:
def ridge(alpha=1.0):
    return make_pipeline(StandardScaler(), Ridge(alpha=alpha))

split = int(len(X_raw) * 0.8)
X_tr, X_te = X_raw.iloc[:split], X_raw.iloc[split:]
y_tr, y_te = y_all.iloc[:split], y_all.iloc[split:]
print("holdout:", X_te.index[0], "→", X_te.index[-1], f"({len(X_te)} rows)")

holdout: 2023-08-09 14:00:00+00:00 → 2023-12-31 23:00:00+00:00 (3466 rows)


**Fix 4 — naive baselines first.** A 24h-ahead model must beat "same hour yesterday" and
"same hour last week" on the same holdout, otherwise the features add nothing.

In [5]:
baselines = pd.Series({
    "naive lag24":  rmse(y_te, X_te["lag24"]),
    "naive lag168": rmse(y_te, X_te["lag168"]),
    "naive roll24": rmse(y_te, X_te["roll24"]),
}).round(1)
baselines

naive lag24     1629.2
naive lag168    1423.5
naive roll24    3806.4
dtype: float64

**Fix 5 — same window, same features when comparing.** The mock compared an untuned model on the
v1 features on a 2023-08→12 holdout against a *tuned* model with two extra (leaked) features
scored by CV over 2022–2023, and called the 42% gap "improvement from tuning". Here every
number is on the same holdout.

In [6]:
untuned = ridge(1.0).fit(X_tr, y_tr)
rmse_untuned = rmse(y_te, untuned.predict(X_te))

leak_tr, leak_te = leaked.iloc[:split], leaked.iloc[split:]
with_leak = ridge(1.0).fit(leak_tr, y_tr)
rmse_leak = rmse(y_te, with_leak.predict(leak_te))

print(f"honest features, alpha=1, holdout RMSE: {rmse_untuned:.1f}")
print(f"with lag1/lag6 (mock),      holdout RMSE: {rmse_leak:.1f}   <- the mock's 'improvement' is this gap")

honest features, alpha=1, holdout RMSE: 953.8
with lag1/lag6 (mock),      holdout RMSE: 538.3   <- the mock's 'improvement' is this gap


**Fix 6 — tune on the training period only, with a time-series CV and a gap.** Shuffled `KFold`
puts neighbouring hours of a highly autocorrelated series in train and test. `TimeSeriesSplit`
respects order; `gap=24` makes sure the last training row's *target* (which the forecaster
would only observe 24h after issuance) is not adjacent to the first test row.

In [7]:
alphas = [0.01, 0.1, 1, 10, 100, 1000]
tscv = TimeSeriesSplit(n_splits=5, gap=24)
grid = GridSearchCV(ridge(), {"ridge__alpha": alphas}, cv=tscv, scoring="neg_root_mean_squared_error")
grid.fit(X_tr, y_tr)
best_alpha = grid.best_params_["ridge__alpha"]
cvres = pd.DataFrame(grid.cv_results_)[["param_ridge__alpha", "mean_test_score", "std_test_score"]]
cvres["mean_test_score"] = -cvres["mean_test_score"]
print("best alpha:", best_alpha)
cvres.round({"mean_test_score": 1, "std_test_score": 1})

best alpha: 100


,param_ridge__alpha,mean_test_score,std_test_score
0,0.01,1099.2,182.1
1,0.10,1099.1,181.9
2,1.00,1098.2,180.3
3,10.00,1092.7,169.0
4,100.00,1079.4,119.4
5,1000.00,1122.5,68.9


How much did `gap` matter here? (Small for Ridge with these features — but it is the principle that is examined.)

In [8]:
for g in (0, 24):
    cv = -cross_val_score(ridge(best_alpha), X_tr, y_tr, cv=TimeSeriesSplit(n_splits=5, gap=g),
                          scoring="neg_root_mean_squared_error")
    print(f"gap={g:<3d} CV RMSE {cv.mean():.1f}")

gap=0   CV RMSE 1067.0
gap=24  CV RMSE 1079.4


**Fix 7 — choose alpha by out-of-sample error, not train fit.** The mock's table showed train RMSE
falling monotonically with alpha and the author picked the smallest. Here we print both and pick by CV.

In [9]:
rows = []
for a in alphas:
    cv = -cross_val_score(ridge(a), X_tr, y_tr, cv=tscv, scoring="neg_root_mean_squared_error")
    m = ridge(a).fit(X_tr, y_tr)
    rows.append({"alpha": a, "train_rmse": rmse(y_tr, m.predict(X_tr)), "cv_rmse": cv.mean(),
                 "holdout_rmse": rmse(y_te, m.predict(X_te))})
pd.DataFrame(rows).set_index("alpha").round(1)

,train_rmse,cv_rmse,holdout_rmse
alpha,,,
0.01,991.7,1099.2,953.8
0.10,991.7,1099.1,953.8
1.00,991.7,1098.2,953.8
10.00,991.8,1092.7,953.9
100.00,998.2,1079.4,958.3
1000.00,1059.5,1122.5,992.6


In [10]:
tuned = ridge(best_alpha).fit(X_tr, y_tr)
rmse_tuned = rmse(y_te, tuned.predict(X_te))
print(f"tuned   holdout RMSE: {rmse_tuned:.1f}")
print(f"untuned holdout RMSE: {rmse_untuned:.1f}")
print(f"gain from tuning:     {(rmse_untuned - rmse_tuned) / rmse_untuned * 100:.2f}%")

tuned   holdout RMSE: 958.3
untuned holdout RMSE: 953.8
gain from tuning:     -0.47%


Coefficients now refer to standardised features and the intercept is the load of the reference hour (h_0).

In [11]:
coefs = pd.Series(tuned.named_steps["ridge"].coef_, index=X_raw.columns)
print("intercept:", round(tuned.named_steps["ridge"].intercept_, 1))
coefs.round(1).sort_values().iloc[list(range(5)) + list(range(-5, 0))]

intercept: 29340.8


temp      -1560.5
dow        -691.2
roll168    -610.9
h_3        -220.1
h_2        -197.4
lag48       734.4
h_17        790.6
h_18        842.3
lag24       873.7
lag168     1141.1
dtype: float64

**Fix 8 — walk-forward without peeking.** `X.loc[:"2023-03"]` *includes* March, so the mock trained on
the month it was predicting. Train strictly before the month's first hour.

**Fix 9 — align by index, never by truncation.** Collect predictions as a Series indexed by time and
compare to `y_all.loc[preds.index]`. The mock compared January-onward predictions against
August-onward actuals and got RMSE 6,910.

**Fix 10 — pool the errors** (RMSE over all OOS hours) rather than averaging monthly RMSEs; with equal
months it barely matters, but pooling is the quantity you actually care about.

In [12]:
months = pd.period_range("2023-01", "2023-12", freq="M")
fold_rmse, pred_parts = [], []
for m in months:
    start = m.start_time.tz_localize("UTC")
    tr_mask = X_raw.index < start
    te_mask = (X_raw.index >= start) & (X_raw.index < m.end_time.tz_localize("UTC"))
    wf = ridge(best_alpha).fit(X_raw[tr_mask], y_all[tr_mask])
    p = pd.Series(wf.predict(X_raw[te_mask]), index=X_raw.index[te_mask])
    pred_parts.append(p)
    fold_rmse.append(rmse(y_all[te_mask], p))

preds = pd.concat(pred_parts)
pooled = rmse(y_all.loc[preds.index], preds)
print(f"pooled walk-forward RMSE 2023: {pooled:.1f}")
print(f"mean of monthly RMSE:          {np.mean(fold_rmse):.1f}")
print(f"naive lag168 over same hours:  {rmse(y_all.loc[preds.index], X_raw.loc[preds.index, 'lag168']):.1f}")
pd.Series(fold_rmse, index=months.astype(str), name="rmse").round(1)

pooled walk-forward RMSE 2023: 989.2
mean of monthly RMSE:          988.7
naive lag168 over same hours:  1587.3


2023-01     993.2
2023-02    1048.3
2023-03     952.3
2023-04     986.4
2023-05    1075.7
2023-06    1001.8
2023-07     990.1
2023-08    1003.9
2023-09     930.3
2023-10     963.2
2023-11     948.6
2023-12     970.7
Name: rmse, dtype: float64

## Honest results

In [13]:
res = pd.Series({
    "naive lag24 (holdout)": baselines["naive lag24"],
    "naive lag168 (holdout)": baselines["naive lag168"],
    "Ridge honest, alpha=1 (holdout)": rmse_untuned,
    f"Ridge honest, tuned alpha={best_alpha} (holdout)": rmse_tuned,
    "Ridge with leaked lag1/lag6 (holdout)": rmse_leak,
    "walk-forward pooled 2023": pooled,
}, name="RMSE MWh").round(1)
res

naive lag24 (holdout)                      1629.2
naive lag168 (holdout)                     1423.5
Ridge honest, alpha=1 (holdout)             953.8
Ridge honest, tuned alpha=100 (holdout)     958.3
Ridge with leaked lag1/lag6 (holdout)       538.3
walk-forward pooled 2023                    989.2
Name: RMSE MWh, dtype: float64